# Calling LLMs from LangChain

Same LCEL pattern for each backend: `prompt | llm | StrOutputParser()`.

You can skip sections you do not have keys or local services for. Keep `.env` values unquoted with no extra spaces.

- [OpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai)
- [OpenRouter](https://docs.langchain.com/oss/python/integrations/chat/openrouter)
- [Hugging Face](https://docs.langchain.com/oss/python/integrations/providers/huggingface)
- [Ollama](https://docs.langchain.com/oss/python/integrations/chat/ollama)

See **Optional dependencies (terminal)** below before the OpenRouter (`ChatOpenRouter`) and Ollama sections.

## Optional dependencies (terminal)

OpenRouter (`ChatOpenRouter`) and Ollama sections need extra packages. OpenAI-direct and OpenRouter-via-`ChatOpenAI` cells only need what is already in the course `requirements.txt`.

From the **repository root**, with the project virtual environment activated:

```bash
pip install -U langchain-openrouter langchain-ollama ollama
```

To snapshot every package in the active venv into `requirements.txt` (e.g. for your own fork):

```bash
pip freeze > requirements.txt
```

Run from the repository root. This overwrites `requirements.txt` with the full environment lockfile.

In [28]:
import os
import getpass
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(override=True)


if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your HF API key: ")

In [18]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful writing assistant."),
    ("user", "{question}"),
])
question = "Write a short paragraph about Dublin, Ireland."

## OpenAI (direct)

Uses `OPENAI_API_KEY` and the default OpenAI API host.

In [ ]:
from langchain_openai import ChatOpenAI

openai_chain = (
    prompt
    | ChatOpenAI(model="gpt-5-nano-2025-08-07", temperature=0.25, max_tokens=512)
    | StrOutputParser()
)
openai_chain.invoke({"question": question})

In [ ]:
# Instruction-tuned OpenAI model
openai_instruct_chain = (
    prompt
    | ChatOpenAI(model="gpt-5-mini-2025-08-07", temperature=0.25, max_tokens=512)
    | StrOutputParser()
)
openai_instruct_chain.invoke({"question": question})

## OpenAI client via OpenRouter endpoint

OpenRouter exposes an [OpenAI-compatible API](https://openrouter.ai/docs). Use your **OpenRouter** key and OpenRouter **model slugs** (e.g. `openai/gpt-4o-mini`), not bare OpenAI model names.

In [19]:
from langchain_openai import ChatOpenAI

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

openrouter_compat_chain = (
    prompt
    | ChatOpenAI(
        model="nvidia/nemotron-3-ultra-550b-a55b:free",
        openai_api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=OPENROUTER_BASE_URL,
        temperature=0.25,
        max_tokens=512,
    )
    | StrOutputParser()
)
openrouter_compat_chain.invoke({"question": question})

'Dublin, the vibrant capital of Ireland, sits at the mouth of the River Liffey on the country’s east coast, blending a rich Viking and medieval history with a youthful, modern energy. The city’s compact center is a tapestry of Georgian architecture, cobblestone streets, and iconic landmarks such as Trinity College—home to the ancient Book of Kells—Dublin Castle, and the twin cathedrals of Christ Church and St. Patrick’s. Renowned globally for its literary legacy as the hometown of Joyce, Yeats, Wilde, and Beckett, Dublin earned UNESCO City of Literature status, a heritage celebrated in its countless historic pubs where traditional music sessions fuel the famous *craic* (fun and conversation). Beyond the tourist trail, neighborhoods like Stoneybatter and Portobello reveal a thriving culinary scene and creative spirit, making Dublin a destination that feels both grandly historic and intimately welcoming.'

## OpenRouter (`ChatOpenRouter`)

Native integration; reads `OPENROUTER_API_KEY` from the environment. Same model slug as above for comparison with the compat client.

In [21]:
from langchain_openrouter import ChatOpenRouter

openrouter_chain = (
    prompt
    | ChatOpenRouter(model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free", temperature=0.25, max_tokens=512)
    | StrOutputParser()
)
openrouter_chain.invoke({"question": question})

'Dublin, the vibrant capital of Ireland, sits on the banks of the River Liffey and blends historic charm with a lively modern spirit. Its cobbled streets are lined with centuries‑old pubs, Georgian architecture, and bustling markets, while cultural landmarks such as Trinity College, the Guinness Storehouse, and the iconic Ha’penny Bridge draw visitors from around the world. Known for its warm hospitality, rich literary heritage, and thriving music scene, Dublin offers a dynamic mix of tradition and contemporary energy that captures the heart of Irish life.'

## Hugging Face Inference API

Serverless inference on Hugging Face. Uses `HUGGINGFACEHUB_API_TOKEN`. If a model fails, pick another from [Inference models](https://huggingface.co/inference/models) or set `provider=` on `HuggingFaceEndpoint`.

In [26]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

HF_MODEL_ID = "prism-ml/Ternary-Bonsai-27B-AWQ-4bit:together"  # change if unavailable on your HF account

hf_llm = ChatHuggingFace(
    llm=HuggingFaceEndpoint(
        repo_id=HF_MODEL_ID,
        max_new_tokens=512,
        temperature=0.25,
    ),
)

hf_chain = prompt | hf_llm | StrOutputParser()
hf_chain.invoke({"question": question})

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ae0c8c9-03bb881c0f7bd73929ef4e5f;0e48fca7-310a-4261-a170-6915d4e9ab9d)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.

## Ollama (local)

Requires [Ollama](https://ollama.com/) running (`ollama serve`) and a pulled model (`ollama pull <name>`). Reads `MODEL` and `OLLAMA_BASE_URL` from `.env`. Large models can be slow on laptops.

In [27]:
from langchain_ollama import ChatOllama

ollama_model = os.getenv("MODEL", "qwen2.5:14b")
ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

ollama_chain = (
    prompt
    | ChatOllama(
        model=ollama_model,
        base_url=ollama_base_url,
        temperature=0.25,
        num_predict=512,
    )
    | StrOutputParser()
)
ollama_chain.invoke({"question": question})

ConnectError: [Errno 61] Connection refused